# Explainability-Guided SegFormer-B0 — Full-Scale Training (Phase 2)

Person 2 (Kalana) GPU run. Same pipeline as Person 3's unexecuted `Phase1/Dinura-Person3/segformer_full_scale_colab.ipynb`, with every write kept under `Phase2/Kalana-Person2/{checkpoints,results}/` — Person 3's and Person 4's Phase 1 folders are import-only.

Split: **3576 / 766 / 766**, seed 42 (identical held-out test set to Chanupa's U-Net baseline). Epochs: **20**. Variants: `vanilla` and `att`.

**Before running:** Runtime → Change runtime type → GPU (T4 or better).


## Step 0: Upload + environment

Upload the repo to Drive (needs `Phase1/Dinura-Person3/`, `Phase1/Lasana-Person4_Evaluation/`, `Phase1/Kalana-Person2/{images,masks}`, and this folder). Adjust `DRIVE_BASE` if the layout differs.


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/DNN-Project")  # ⬅️ adjust
    HERE = DRIVE_BASE / "Phase2" / "Kalana-Person2"
    print("Colab: Drive mounted")
else:
    HERE = Path.cwd()
    if HERE.name != "Kalana-Person2":
        candidate = Path("Phase2/Kalana-Person2").resolve()
        if candidate.is_dir():
            HERE = candidate
    print("Local run: skipping Google Drive mount")

sys.path.insert(0, str(HERE))
print("HERE =", HERE, "-> exists:", HERE.is_dir())


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "thop", "tqdm"])
print("deps ready")


In [ ]:
from paths import (
    DATA_IMG_DIR, DATA_MASK_DIR, PERSON3_DIR, PERSON4_DIR,
    CKPT_DIR, RESULTS_DIR, N_TRAIN, N_VAL, N_TEST, SEED, EPOCHS,
    add_teammate_paths, apply_data_dirs, ensure_output_dirs,
)

add_teammate_paths()
apply_data_dirs()
ensure_output_dirs()

print("Person 3 package:", PERSON3_DIR, "->", PERSON3_DIR.is_dir())
print("Person 4 eval:   ", PERSON4_DIR, "->", PERSON4_DIR.is_dir())
print("images:          ", DATA_IMG_DIR, "->", DATA_IMG_DIR.is_dir())
print("masks:           ", DATA_MASK_DIR, "->", DATA_MASK_DIR.is_dir())
print("checkpoints out: ", CKPT_DIR)
print("results out:     ", RESULTS_DIR)
print(f"split {N_TRAIN}/{N_VAL}/{N_TEST}  seed={SEED}  epochs={EPOCHS}")
assert PERSON3_DIR.is_dir(), "Upload Phase1/Dinura-Person3 next to this repo copy"
assert DATA_MASK_DIR.is_dir(), "Upload Phase1/Kalana-Person2/masks"
assert DATA_IMG_DIR.is_dir(), "Upload Phase1/Kalana-Person2/images"


## Step 1: Device check


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Full-scale 20-epoch x 2 variants will not finish in a reasonable time.")


## Step 2: Split preflight

Confirms the 3576/766/766 seed-42 lists match Chanupa's U-Net split algorithm before any GPU time is spent. Same test as `tests/test_split_identity.py`.


In [ ]:
import runpy
runpy.run_path(str(HERE / "tests" / "test_split_identity.py"), run_name="__main__")


## Step 3: Full-scale training — both variants

Reuses the same `train_variant` logic as Person 3's smoke trainer (pretrained MiT-B0, Attention Consistency Loss, double backprop). Checkpoints write here, not under `Dinura-Person3/`.

Resume: if `segformer_b0_{variant}_best.pt` already exists, that variant is skipped.


In [ ]:
import train_full_scale as T
from paths import CKPT_DIR, BATCH_SIZE, LR, LAMBDA2, SIGMA, ATT_MODE, SEED, EPOCHS, N_TRAIN, N_VAL, N_TEST

class Args:
    n_train, n_val, n_test = N_TRAIN, N_VAL, N_TEST
    epochs = EPOCHS
    batch_size = BATCH_SIZE
    lr = LR
    lambda2 = LAMBDA2
    sigma = SIGMA
    att_mode = ATT_MODE
    seed = SEED

args = Args()
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

for variant in ("vanilla", "att"):
    best = CKPT_DIR / f"segformer_b0_{variant}_best.pt"
    if best.exists():
        print(f"SKIP train {variant}: {best.name} already exists")
        continue
    T.train_variant(variant, args)


## Step 4: Evaluate both checkpoints (Dice / IoU / F1 / AAMO / efficiency)

Same held-out 766 images as Step 3. Person 4's `metrics.py` / `aamo.py` / `efficiency.py` — numbers comparable to the U-Net row. Writes **only** `Phase2/Kalana-Person2/results/` (does not copy into Lasana's folder).


In [ ]:
import eval_full_scale as E

class EvalArgs:
    n_train, n_val, n_test = N_TRAIN, N_VAL, N_TEST
    seed = SEED

rows = []
for variant in ("vanilla", "att"):
    rows.append(E.evaluate_variant(variant, EvalArgs()))
print("\nFull-scale rows:")
for r in rows:
    print(f"  {r['model']}: dice={r['dice']} iou={r['iou']} aamo={r['aamo']}")


## Step 5: Qualitative attention-drift figures


In [ ]:
import generate_full_scale_figures as G
import sys
sys.argv = ["generate_full_scale_figures.py", "--n", "3"]
G.main()


## Step 6: Confirm outputs stayed in this folder

Ping Dhinanjaya with `results/baseline_comparison.md` and the `attention_drift_*_full_scale.png` figures. Paper abstract / Table 1 / Figure 2 / smoke-scale caveats are his follow-up — not edited here.


In [ ]:
from paths import CKPT_DIR, RESULTS_DIR, HERE as P2
print("outputs under", P2)
for p in sorted(CKPT_DIR.glob("*.pt")):
    print(" ckpt", p.name, p.stat().st_size)
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file():
        print(" result", p.relative_to(RESULTS_DIR))
